# 08 · Any molecule: XYZ → certified effective Hamiltonian → symbolic language model

1. Analyse molecules from geometry alone: point group, Γ_vib, spin statistics.
2. Plan a certified zeroth-order model (rotational constants from the geometry).
3. Let the symbolic language model propose the next terms; grow the model on data with `run`.

In [ ]:
import math, numpy as np
import riss_drucker as rd
from riss_drucker.molecule import Molecule, run
from riss_drucker import build as B, autobuild as AB
from riss_drucker.symbolic_lm import SymbolicLM, default_degrees
xyz = {"H2O": "O 0 0 0.1173\nH 0 0.7572 -0.4692\nH 0 -0.7572 -0.4692",
       "NH3": "N 0 0 0.1162\nH 0 0.9377 -0.2712\nH 0.8121 -0.4689 -0.2712\nH -0.8121 -0.4689 -0.2712",
       "CH4": "C 0 0 0\nH 0.6276 0.6276 0.6276\nH 0.6276 -0.6276 -0.6276\nH -0.6276 0.6276 -0.6276\nH -0.6276 -0.6276 0.6276"}
for k, v in xyz.items():
    m = Molecule.from_xyz(v)
    a = m.analyse()
    print(k, a["point_group"], a["rotor"], {x: round(y, 3) for x, y in a["rotational_constants_cm"].items()},
          a["gamma_vib"], a["spin_weights"])

In [ ]:
w = Molecule.from_xyz(xyz["H2O"])
p = w.plan(frequencies={"A1": [1595.0, 3657.0], "B1": [3756.0]}, J=[0, 1, 2])
bm = B.compile_build(p["spec"])
print(p["spec"].terms)
print(len(bm.certificates), "certificates passed;", len(bm.blocks), "blocks")
print(SymbolicLM.default().generate(p["spec"], default_degrees(p["spec"]), top=6,
                                    frequencies={"1": 1595.0, "2": 3657.0, "3": 3756.0}))

In [ ]:
p0 = w.plan(J=[0, 1, 2, 3, 4], nmax=0)
truth = B.compile_build(p0["spec"].with_terms(p0["spec"].terms + ["DK: sym(Jz Jz Jz Jz) + h.c."]))
th = {**p0["theta"], "DK": -2e-3}
rng = np.random.default_rng(2)
data = [AB.LevelDatum(b.two_j / 2, b.irrep, i, e + 1e-4 * rng.standard_normal(), 1e-4, b.polyad)
        for b, (ws, _) in zip(truth.blocks, truth.eig(th)) for i, e in enumerate(ws)]
out = run(w, data, J=[0, 1, 2, 3, 4], nmax=0, max_steps=3)
for s in out["result"].steps:
    print(s.accepted, s.term, s.reason[:50])
print("chi2/dof", round(out["result"].fit.reduced_chi2, 2))